# 9 WorkFlow Analista Jr

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Sep 07 11:26:05 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671174,35.9,1479531,79.1,1479531,79.1
Vcells,1242565,9.5,8388608,64.0,1978697,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros

In [4]:
PARAM <- list()
PARAM$semilla_primigenia <- 100313

PARAM$experimento <- 9102
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

#### Carpeta del Experimento

In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_deflacion()
fin drift_deflacion()


In [26]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"

In [27]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

##### Inicio GA

#### =================================================================
#### PROBLEMA #10: Feature Engineering Intra-mes - Algoritmo Genetico
#### Modalidad Analista Jr / Sr - Grupo A
####
#### HIPOTESIS EXPERIMENTAL:
####   "Un algoritmo genetico que vaya generando las combinaciones de
####    variables y descarte aquellas que no sean de utilidad debe ser
####    mucho mejor que las variables que se nos puedan ocurrir o que
####    la bibliografia acerca del sector financiero pueda plantear."
####
#### ARQUITECTURA (2 etapas, cada una con su paquete):
####  - Etapa 1 - gramEvol: GENERA un pool de formulas candidatas
####             (evolucion gramatical, fitness barato = correlacion)
####  - Etapa 2 - GA:       SELECCIONA el subconjunto optimo del pool
####             (algoritmo genetico binario, fitness = AUC real)
#### IMPORTANTE: en este punto del pipeline TODAVIA NO EXISTEN
####   fold_train, PARAM$training, PARAM$validate (se crean recien en
####   9.3.2.1 Training Strategy). Por eso ambas etapas usan un split
####   LOCAL propio, cuidando siempre de no tocar el mes 202107 (que
####   mas adelante sera el validate oficial del pipeline).
#### =================================================================

In [28]:
# Paso 0: Instalar librerias GA

if( !require("GA")) install.packages("GA")
require("GA")
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")
install.packages("gramEvol", repos = "https://cloud.r-project.org")
library(gramEvol)

# CELDA 0 - Semillas del experimento (multi-semilla + trazabilidad)
PARAM$semillas_GA <- c(100313, 214351, 357913, 701819, 900001)

# indice de la corrida actual (1..length(PARAM$semillas_GA))
k <- 1

stopifnot(k >= 1, k <= length(PARAM$semillas_GA))
PARAM$semilla_primigenia <- PARAM$semillas_GA[k]

cat("=== CORRIDA", k, "de", length(PARAM$semillas_GA),
    "| semilla_primigenia =", PARAM$semilla_primigenia, "===\n")

 

Loading required package: GA

Loading required package: foreach

Loading required package: iterators

Package 'GA' version 3.2.5
Type 'citation("GA")' for citing this R package in publications.


Attaching package: ‘GA’


The following object is masked from ‘package:utils’:

    de


Loading required package: lightgbm

Installing package into ‘/home/ds/.local/lib/R/site-library’
(as ‘lib’ is unspecified)



=== CORRIDA 1 de 5 | semilla_primigenia = 100313 ===


In [29]:
# CELDA 1 - Variables candidatas (terminales de la gramatica)
 
# clase01: fallback por si esta celda corre antes de que el pipeline
# oficial la calcule mas adelante
if (!"clase01" %in% colnames(dataset)) {
  dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1, 0)]
}

excluir_GA <- c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "azar")
candidatas_GA <- setdiff(colnames(dataset), excluir_GA)

# me quedo solo con columnas numericas
son_numericas <- sapply(dataset[, candidatas_GA, with = FALSE], is.numeric)
candidatas_GA <- candidatas_GA[son_numericas]

# excluyo variables de fecha (ver nota arriba)
patron_fechas <- "^(f|.*_f|.*fecha)"
candidatas_GA <- candidatas_GA[!grepl(patron_fechas, candidatas_GA, ignore.case = TRUE)]

# limite de seguridad para no explotar el espacio de busqueda
MAX_TERMINALES <- 40
if (length(candidatas_GA) > MAX_TERMINALES) {
  set.seed(PARAM$semilla_primigenia)
  candidatas_GA <- sample(candidatas_GA, MAX_TERMINALES)
}

cat("Variables candidatas para el algoritmo genetico:", length(candidatas_GA), "\n")
print(candidatas_GA)

Variables candidatas para el algoritmo genetico: 40 
 [1] "ccallcenter_transacciones"     "ctarjeta_master"              
 [3] "mcuentas_saldo"                "cdescubierto_preacordado"     
 [5] "internet"                      "Visa_mconsumototal"           
 [7] "mcaja_ahorro"                  "Visa_mlimitecompra"           
 [9] "mcuenta_corriente"             "mprestamos_personales"        
[11] "mtarjeta_master_consumo"       "cliente_antiguedad"           
[13] "Visa_status"                   "Visa_cconsumos"               
[15] "mpayroll"                      "cprestamos_personales"        
[17] "mtarjeta_visa_consumo"         "Master_mpagominimo"           
[19] "mcomisiones_mantenimiento"     "cliente_edad"                 
[21] "Visa_msaldototal"              "ccomisiones_mantenimiento"    
[23] "Master_cconsumos"              "Master_mconsumototal"         
[25] "cproductos"                    "mrentabilidad_annual"         
[27] "Master_mlimitecompra"          "chomebanking

In [30]:
# =================================================================
# CELDA 2 - Operadores protegidos + definicion de la gramatica
# =================================================================
# Que hacemos: definimos que combinaciones son sintacticamente
# validas (la gramatica), usando operadores "protegidos" para evitar
# errores por division por cero o logaritmo/raiz de negativos
# (variables monetarias pueden ser negativas; hay columnas en NA
# por el Catastrophe Analysis del pipeline).
#
# NOTA IMPORTANTE: la funcion correcta para crear la gramatica es
# CreateGrammar(), NO GrammarDef() (error comun, GrammarDef no existe
# en este paquete).

div_protegida <- function(a, b) {
  b_seguro <- ifelse(abs(b) < 1e-6, 1e-6, b)
  a / b_seguro
}
log_seguro  <- function(x) log(abs(x) + 1)
raiz_segura <- function(x) sqrt(abs(x))

ruleDef <- list(
  expr = grule(op(expr, expr), func(expr), var),
  op   = grule(`+`, `-`, `*`, div_protegida),
  func = grule(log_seguro, raiz_segura),
  var  = do.call(grule, lapply(candidatas_GA, as.symbol))
)

grammarDef <- CreateGrammar(ruleDef)

# prueba de humo: confirmo que la gramatica genera las 3 ramas
# (variable sola, operador entre dos expr, funcion aplicada)
set.seed(42)
cat("\nPrueba de la gramatica (deberia verse variedad):\n")
for (i in 1:5) {
  cat(" ", deparse(GrammarRandomExpression(grammarDef)[[1]]), "\n")
}



Prueba de la gramatica (deberia verse variedad):
  raiz_segura(ccomisiones_mantenimiento) - Visa_mconsumototal 
  raiz_segura(mcomisiones) 
  Visa_mlimitecompra 
  Master_mconsumototal 
  mrentabilidad_annual 


In [31]:
# =================================================================
# CELDA 3 - Funcion de fitness para gramEvol (proxy BARATO)
# =================================================================
# Que hacemos: en vez de entrenar un LightGBM por cada formula
# candidata (inviable: serian miles de entrenamientos), usamos como
# proxy el AUC UNIVARIADO de la formula contra clase01.
#
# POR QUE AUC (y no Pearson ni Spearman):
#   - AUC mide poder de RANKING (ordenar clientes por riesgo de baja),
#     que es exactamente lo que valora un modelo de churn y lo que
#     optimiza el resto del pipeline (grid search oficial usa AUC, y
#     la Etapa 2 con GA + LightGBM tambien). Asi las dos etapas del
#     genetico hablan la MISMA metrica que el notebook oficial.
#   - AUC NO asume linealidad (a diferencia de Pearson), asi que capta
#     relaciones no lineales que un modelo de arboles si aprovecha.
#   - Sigue siendo BARATO: el AUC de una sola variable equivale al
#     estadistico de Mann-Whitney (rank-sum), no requiere entrenar
#     ningun modelo.
#   - Spearman NO sirve aca: es invariante a transformaciones monotonas
#     (log, sqrt darian el mismo fitness que la variable sin transformar),
#     con lo que gramEvol no tendria incentivo a probar funciones.
#
# Split LOCAL: el fitness de gramEvol se calcula SOLO sobre el mismo
# tramo que luego sera TRAIN del GA (foto_mes < 202105). Los meses de
# validacion local del GA (202105, 202106) NO participan en la
# construccion del fitness -> asi se evita el leakage de seleccion
# (elegir formulas mirando el target de los meses de validacion, lo
# que inflaria el AUC/ganancia reportados). El mes 202107 (validate
# oficial) tampoco se toca.

train_rows_gramevol <- dataset[foto_mes < 202105]

# AUC univariado via Mann-Whitney (rank-sum), sin entrenar modelos.
# Formula: AUC = (U) / (n_pos * n_neg), donde U se obtiene de la suma
# de rangos de la clase positiva. Devuelve un valor en [0, 1].
target_gramevol <- train_rows_gramevol$clase01

auc_univariado <- function(valores, target) {
  # blindaje: valores y target deben tener la misma longitud. Si eval
  # devolvio un escalar reciclado o un largo raro, abortamos a 0.5.
  if (length(valores) != length(target)) return(0.5)

  ok <- is.finite(valores) & !is.na(target)
  v <- valores[ok]; y <- target[ok]

  # si tras filtrar quedan muy pocos casos o una sola clase, no hay AUC
  if (length(v) < 2L) return(0.5)
  n_pos <- sum(y == 1); n_neg <- sum(y == 0)
  if (n_pos == 0L || n_neg == 0L) return(0.5)

  r <- rank(v)                         # rangos (promedia empates)
  suma_rangos_pos <- sum(r[y == 1])
  auc <- (suma_rangos_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

  # red de seguridad final: cualquier cosa no finita -> 0.5 (sin poder)
  if (!is.finite(auc)) return(0.5)
  auc
}

evaluar_formula <- function(expr) {
  valores <- tryCatch(
    eval(expr, envir = train_rows_gramevol),
    error = function(e) rep(NA_real_, nrow(train_rows_gramevol))
  )

  # fuerzo tipo numerico: si eval devolvio logical/integer/otro, lo
  # convierto; si no es convertible, aborto a 0.5
  valores <- suppressWarnings(as.numeric(valores))

  n <- nrow(train_rows_gramevol)
  # caso escalar reciclado (longitud 1): la formula es constante -> sin poder
  if (length(valores) == 1L) return(0.5)
  if (length(valores) != n) return(0.5)
  if (all(!is.finite(valores))) return(0.5)

  desvio <- sd(valores, na.rm = TRUE)
  if (is.na(desvio) || desvio == 0) return(0.5)   # constante -> sin poder

  auc_univariado(valores, target_gramevol)
}

costFunction <- function(expr) {
  # gramEvol MINIMIZA por defecto. Una variable y su negativo ordenan
  # igual de bien (AUC y 1-AUC son equivalentes en poder de ranking),
  # asi que el "poder" de una formula es cuan lejos esta de 0.5.
  # Devuelvo -abs(AUC - 0.5): mas poder de ranking = mas negativo = mejor.
  auc <- evaluar_formula(expr)
  costo <- -abs(auc - 0.5)
  # RED DE SEGURIDAD: gramEvol ABORTA si el cost es NA/NaN/Inf.
  # Cualquier resultado no finito lo mapeamos a 0 (peor cost posible,
  # = formula sin poder de ranking), para que la evolucion continue.
  if (!is.finite(costo)) return(0)
  costo
}

In [32]:
# =================================================================
# CELDA 4 - Etapa 1: N corridas de gramEvol (arma el POOL)
# =================================================================
# Que hacemos: corremos GrammaticalEvolution() varias veces, cada
# vez con una semilla distinta, porque UNA sola corrida converge a
# una unica region del espacio de formulas. Para tener un pool
# variado (que es lo que la Etapa 2 va a filtrar), necesitamos
# multiples corridas independientes.
#
# Tiempo esperado: con fitness AUC univariado (Mann-Whitney, sin
# entrenar modelos), cada corrida de 300 iteraciones tarda ~45-60
# segundos en esta VM (8 vCPU). 20 corridas -> aprox 15-18 minutos.

N_CORRIDAS <- 20

pool_formulas <- vector("list", N_CORRIDAS)
pool_fitness  <- numeric(N_CORRIDAS)

for (i in 1:N_CORRIDAS) {
  semilla_i <- PARAM$semilla_primigenia + i
  set.seed(semilla_i)

  resultado_GE <- GrammaticalEvolution(
    grammarDef,
    costFunction,
    iterations       = 300,
    terminationCost  = -0.15,   # corta si abs(AUC-0.5) >= 0.15 (AUC >= 0.65)
    max.depth        = 4,
    disable.warnings = TRUE
  )

  pool_formulas[[i]] <- resultado_GE$best$expressions[[1]]
  # el cost es -abs(AUC-0.5); lo reconvierto a AUC interpretable (>= 0.5)
  pool_fitness[i]    <- 0.5 + (-resultado_GE$best$cost)

  cat("Corrida", i, "/", N_CORRIDAS,
      " | AUC univ:", round(pool_fitness[i], 4),
      " | formula:", deparse(pool_formulas[[i]]), "\n")
}

cat("\n=== POOL COMPLETO:", length(pool_formulas), "formulas ===\n")


Corrida 1 / 20  | AUC univ: 0.6596  | formula: ccomisiones_mantenimiento - raiz_segura(Master_mconsumototal) 
Corrida 2 / 20  | AUC univ: 0.6572  | formula: div_protegida(cpayroll_trx, Master_mconsumototal) 
Corrida 3 / 20  | AUC univ: 0.8115  | formula: raiz_segura(mcaja_ahorro - Master_cconsumos) 
Corrida 4 / 20  | AUC univ: 0.6525  | formula: (ccallcenter_transacciones + Master_mfinanciacion_limite) * Master_mconsumototal 
Corrida 5 / 20  | AUC univ: 0.7108  | formula: log_seguro(log_seguro(Master_cconsumos - thomebanking)) * ctarjeta_visa_transacciones 
Corrida 6 / 20  | AUC univ: 0.7221  | formula: Visa_cconsumos * log_seguro(mcomisiones) 
Corrida 7 / 20  | AUC univ: 0.6675  | formula: raiz_segura(raiz_segura(Master_cconsumos * ctarjeta_master)) 
Corrida 8 / 20  | AUC univ: 0.6904  | formula: Visa_cconsumos * log_seguro(Master_mpagominimo + mtransferencias_recibidas) 
Corrida 9 / 20  | AUC univ: 0.7144  | formula: Visa_mconsumototal + log_seguro(Visa_msaldototal - raiz_segura(mact

In [33]:
# =================================================================
# CELDA 5 - Incorporo el pool completo como columnas del dataset
# =================================================================
for (i in seq_along(pool_formulas)) {
  nombre_col <- paste0("GA_var", sprintf("%02d", i))
  dataset[, (nombre_col) := tryCatch(
    eval(pool_formulas[[i]], envir = .SD),
    error = function(e) NA_real_
  ), .SDcols = candidatas_GA]
}

cols_GA <- paste0("GA_var", sprintf("%02d", 1:length(pool_formulas)))
cat("Columnas GA_var creadas:", length(cols_GA), "\n")



Columnas GA_var creadas: 20 


In [34]:
# =================================================================
# CELDA 6 - Deduplicacion (formulas funcionalmente identicas)
# =================================================================
# Que hacemos: distintas formulas (sintacticamente) pueden terminar
# siendo la MISMA variable en la practica (ej: composiciones de
# funciones monotonas de la misma variable base dan correlacion 1
# entre si). No tiene sentido que GA "elija" entre copias identicas
# -> las sacamos con un filtro deterministico ANTES de pasarle el
# pool a GA, para no desperdiciar busqueda evolutiva en algo trivial.

# la matriz de correlacion para deduplicar se calcula sobre el mismo
# tramo de train que el fitness (foto_mes < 202105), para no medir
# redundancia usando meses de validacion local
matriz_corr <- cor(
  dataset[foto_mes < 202105, ..cols_GA],
  use = "pairwise.complete.obs",
  method = "pearson"
)

UMBRAL_DUPLICADO <- 0.999
ya_marcadas <- rep(FALSE, length(cols_GA))
grupos_duplicados <- list()

for (i in 1:(length(cols_GA) - 1)) {
  if (ya_marcadas[i]) next
  grupo <- cols_GA[i]
  for (j in (i + 1):length(cols_GA)) {
    if (ya_marcadas[j]) next
    r <- matriz_corr[i, j]
    if (!is.na(r) && abs(r) >= UMBRAL_DUPLICADO) {
      grupo <- c(grupo, cols_GA[j])
      ya_marcadas[j] <- TRUE
    }
  }
  if (length(grupo) > 1) grupos_duplicados[[length(grupos_duplicados) + 1]] <- grupo
}

descartar <- unlist(lapply(grupos_duplicados, function(g) g[-1]))
cols_GA_dedup <- setdiff(cols_GA, descartar)

cat("Grupos de duplicados encontrados:\n"); print(grupos_duplicados)
cat("Columnas antes de deduplicar:", length(cols_GA), "\n")
cat("Columnas despues de deduplicar:", length(cols_GA_dedup), "\n")



Grupos de duplicados encontrados:
[[1]]
[1] "GA_var01" "GA_var16"

[[2]]
[1] "GA_var09" "GA_var12" "GA_var19"

Columnas antes de deduplicar: 20 
Columnas despues de deduplicar: 17 


In [35]:
# =================================================================
# CELDA 7 - Etapa 2: Setup de GA para SELECCION (fitness = AUC real)
# =================================================================
# Que hacemos: a diferencia de la Etapa 1 (fitness barato), aca el
# pool ya es chico (17 candidatas, no miles), asi que SI podemos
# pagar el costo de entrenar un LightGBM liviano por cada individuo.
#
# Split LOCAL para esta etapa (distinto del de gramEvol, pero
# igual de cuidadoso en no tocar 202107):
#   train: foto_mes < 202105
#   valid: foto_mes en {202105, 202106}

if (!require("GA")) install.packages("GA", repos = "https://cloud.r-project.org")
library(GA)
library(lightgbm)
library(ROCR)

# variables manuales del docente (FE_intra_manual, seccion 9.3.1.3).
# FALLBACK: si el bloque genetico se corre antes de que esas celdas
# se hayan ejecutado, las creamos aca (misma definicion del notebook
# oficial) para no depender del orden de ejecucion de celdas.
if (!"kmes" %in% colnames(dataset) && "foto_mes" %in% colnames(dataset)) {
  dataset[, kmes := foto_mes %% 100]
}
if (!"mpayroll_sobre_edad" %in% colnames(dataset) &&
    all(c("mpayroll", "cliente_edad") %in% colnames(dataset))) {
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]
}

campos_base_manual <- c("kmes", "mpayroll_sobre_edad")
campos_base_manual <- campos_base_manual[campos_base_manual %in% colnames(dataset)]

stopifnot(
  "No hay variables base manuales (kmes / mpayroll_sobre_edad) ni se pudieron crear" =
    length(campos_base_manual) > 0
)
cat("campos_base_manual:", paste(campos_base_manual, collapse = ", "), "\n")

filas_train_local <- dataset$foto_mes < 202105
filas_valid_local  <- dataset$foto_mes %in% c(202105, 202106)

clase_train_local <- dataset$clase01[filas_train_local]
clase_valid_local <- dataset$clase01[filas_valid_local]

# guardo tambien la clase_ternaria ORIGINAL del fold de validacion,
# la vamos a necesitar en la Celda 9 para calcular GANANCIA real
# (la funcion de ganancia usa BAJA+2 especificamente, no clase01)
clase_ternaria_valid_local <- dataset$clase_ternaria[filas_valid_local]

fitness_GA_seleccion <- function(cromosoma) {

  seleccionadas <- cols_GA_dedup[cromosoma == 1]
  campos <- unique(c(campos_base_manual, seleccionadas))

  # BUG QUE YA RESOLVIMOS: el chequeo de "nada seleccionado" debe
  # mirar 'campos' completo (incluye las manuales), no solo
  # 'seleccionadas' (las GA_var) -> si no, el cromosoma de baseline
  # (todo ceros en GA_var) devuelve 0.5 hardcodeado sin entrenar
  # nada, en vez del AUC real del modelo con solo las manuales.
  # (mantengo solo columnas que realmente existan, por las dudas)
  campos <- campos[campos %in% colnames(dataset)]
  if (length(campos) == 0) return(0.5)

  # blindaje: LightGBM aborta con 'num_data > 0' si el train queda
  # vacio. Verifico que haya filas de train y validacion antes de armar
  # el dataset. Si algun fold esta vacio, devuelvo 0.5 (sin senal).
  if (sum(filas_train_local) == 0 || sum(filas_valid_local) == 0) return(0.5)

  dtrain <- lgb.Dataset(
    data  = data.matrix(dataset[filas_train_local, campos, with = FALSE]),
    label = clase_train_local
  )
  modelo <- lgb.train(
    data = dtrain,
    params = list(objective = "binary", metric = "auc", verbosity = -1,
                  max_bin = 31, learning_rate = 0.1,
                  num_leaves = 31, feature_fraction = 0.8),
    nrounds = 50, verbose = -1
  )
  pred <- predict(modelo, data.matrix(dataset[filas_valid_local, campos, with = FALSE]))

  as.numeric(ROCR::performance(
    ROCR::prediction(pred, clase_valid_local), "auc"
  )@y.values[[1]])
}


campos_base_manual: kmes, mpayroll_sobre_edad 


In [36]:
# =================================================================
# CELDA 8 - Baseline real + corrida completa de GA
# =================================================================
cromosoma_baseline <- rep(0, length(cols_GA_dedup))
auc_baseline <- fitness_GA_seleccion(cromosoma_baseline)
cat("AUC baseline (solo variables manuales del docente):", round(auc_baseline, 4), "\n")

set.seed(PARAM$semilla_primigenia)
resultado_GA_seleccion <- ga(
  type      = "binary",
  fitness   = fitness_GA_seleccion,
  nBits     = length(cols_GA_dedup),
  popSize   = 20,
  maxiter   = 20,
  run       = 8,
  pmutation = 0.1,
  seed      = PARAM$semilla_primigenia,
  monitor   = TRUE
)

mejor_cromosoma <- as.vector(resultado_GA_seleccion@solution[1, ])
vars_ganadoras  <- cols_GA_dedup[mejor_cromosoma == 1]

cat("\nAUC baseline:        ", round(auc_baseline, 4), "\n")
cat("AUC con seleccion GA: ", round(resultado_GA_seleccion@fitnessValue, 4), "\n")
cat("Variables seleccionadas (", length(vars_ganadoras), "): ",
    paste(vars_ganadoras, collapse = ", "), "\n", sep = "")


AUC baseline (solo variables manuales del docente): 0.7511 
GA | iter = 1 | Mean = 0.8670306 | Best = 0.8752627
GA | iter = 2 | Mean = 0.8716466 | Best = 0.8753214
GA | iter = 3 | Mean = 0.8708424 | Best = 0.8753214
GA | iter = 4 | Mean = 0.8715582 | Best = 0.8761547
GA | iter = 5 | Mean = 0.8730818 | Best = 0.8761547
GA | iter = 6 | Mean = 0.8729283 | Best = 0.8764995
GA | iter = 7 | Mean = 0.8730685 | Best = 0.8764995
GA | iter = 8 | Mean = 0.8725968 | Best = 0.8764995
GA | iter = 9 | Mean = 0.8734156 | Best = 0.8764995
GA | iter = 10 | Mean = 0.8732029 | Best = 0.8764995
GA | iter = 11 | Mean = 0.8731807 | Best = 0.8764995
GA | iter = 12 | Mean = 0.8728865 | Best = 0.8765162
GA | iter = 13 | Mean = 0.8732205 | Best = 0.8765162
GA | iter = 14 | Mean = 0.8730916 | Best = 0.8765162
GA | iter = 15 | Mean = 0.8731731 | Best = 0.8765162
GA | iter = 16 | Mean = 0.8741738 | Best = 0.8765162
GA | iter = 17 | Mean = 0.8740566 | Best = 0.8765162
GA | iter = 18 | Mean = 0.8745747 | Best = 0.876

In [37]:
# =================================================================
# CELDA 9 - Funcion de GANANCIA oficial + comparacion real
# =================================================================
# ganancia = 975000*BAJA+2 - 25000*(BAJA+1 + CONTINUA), en MILLONES $
# OJO: se calcula sobre clase_ternaria ORIGINAL, no sobre clase01
# (BAJA+1 se usa para entrenar pero NO paga premio en la ganancia real)

ganancia_evaluacion <- function(probabilidades, clase_ternaria_real) {
  tbl <- data.table(prob = probabilidades, clase = clase_ternaria_real)
  setorder(tbl, -prob)
  tbl[, ganancia_individual := fifelse(clase == "BAJA+2", 975000, -25000)]
  tbl[, ganancia_acumulada := cumsum(ganancia_individual)]
  max(tbl$ganancia_acumulada) / 1e6
}

# --- baseline ---
dtrain_base <- lgb.Dataset(
  data  = data.matrix(dataset[filas_train_local, campos_base_manual, with = FALSE]),
  label = clase_train_local
)
modelo_base <- lgb.train(
  data = dtrain_base,
  params = list(objective = "binary", metric = "auc", verbosity = -1,
                max_bin = 31, learning_rate = 0.1, num_leaves = 31, feature_fraction = 0.8),
  nrounds = 50, verbose = -1
)
pred_base <- predict(modelo_base, data.matrix(dataset[filas_valid_local, campos_base_manual, with = FALSE]))
ganancia_base <- ganancia_evaluacion(pred_base, clase_ternaria_valid_local)

# --- con las variables ganadoras de GA ---
campos_GA <- c(campos_base_manual, vars_ganadoras)
dtrain_GA <- lgb.Dataset(
  data  = data.matrix(dataset[filas_train_local, campos_GA, with = FALSE]),
  label = clase_train_local
)
modelo_GA <- lgb.train(
  data = dtrain_GA,
  params = list(objective = "binary", metric = "auc", verbosity = -1,
                max_bin = 31, learning_rate = 0.1, num_leaves = 31, feature_fraction = 0.8),
  nrounds = 50, verbose = -1
)
pred_GA <- predict(modelo_GA, data.matrix(dataset[filas_valid_local, campos_GA, with = FALSE]))
ganancia_GA <- ganancia_evaluacion(pred_GA, clase_ternaria_valid_local)

# --- verificacion de sanidad: techo teorico ---
n_baja2_valid <- sum(clase_ternaria_valid_local == "BAJA+2")
ganancia_maxima_teorica <- (n_baja2_valid * 975000) / 1e6

cat("\n=== RESULTADO FINAL (split local, LightGBM liviano) ===\n")
cat("Ganancia baseline (millones $):        ", round(ganancia_base, 2), "\n")
cat("Ganancia con variables GA (millones $):", round(ganancia_GA, 2), "\n")
cat("Ganancia maxima teorica (millones $):   ", round(ganancia_maxima_teorica, 2), "\n")
cat("% techo teorico - baseline:             ", round(100*ganancia_base/ganancia_maxima_teorica, 1), "%\n")
cat("% techo teorico - con GA:               ", round(100*ganancia_GA/ganancia_maxima_teorica, 1), "%\n")



=== RESULTADO FINAL (split local, LightGBM liviano) ===
Ganancia baseline (millones $):         -0.03 
Ganancia con variables GA (millones $): 73.72 
Ganancia maxima teorica (millones $):    390 
% techo teorico - baseline:              0 %
% techo teorico - con GA:                18.9 %


In [38]:

# =================================================================
# CELDA 10 - Incorporacion DEFINITIVA al dataset (limpieza final)
# =================================================================
# Dejamos SOLO las variables seleccionadas por GA en el dataset;
# borramos el resto del pool para no cargarle trabajo de mas a
# FEhist (que laguea/deltea TODAS las columnas del dataset).

todas_GA_var <- colnames(dataset)[grepl("^GA_var", colnames(dataset))]
descartar_definitivamente <- setdiff(todas_GA_var, vars_ganadoras)
dataset[, (descartar_definitivamente) := NULL]

cat("\nColumnas GA_var finales en el dataset:", sum(grepl("^GA_var", colnames(dataset))), "\n")

# guardo las formulas en texto plano para las slides del informe
formulas_finales <- data.table(
  variable = vars_ganadoras,
  formula  = sapply(vars_ganadoras, function(v) {
    idx <- as.integer(gsub("GA_var", "", v))
    deparse(pool_formulas[[idx]], width.cutoff = 500L)
  })
)
fwrite(formulas_finales, file = "GA_formulas_ganadoras_problema10.txt", sep = "\t")
print(formulas_finales)


# --- LOG DE TRAZABILIDAD (append, una fila por corrida/semilla) -----
# Que hacemos: cada corrida agrega UNA fila a un log acumulativo. Asi,
# aunque el disparo sea manual (cambiar 'k' y correr), queda prueba
# automatica de que semilla produjo que resultado. Sirve para el
# informe/video y para auditar que las 5 corridas usaron las 5
# semillas correctas (sin repetir ninguna).
log_corrida <- data.table(
  timestamp        = format(Sys.time(), "%Y-%m-%d %H:%M:%S"),
  corrida_k        = k,
  semilla          = PARAM$semilla_primigenia,
  n_ganadoras      = length(vars_ganadoras),
  auc_baseline     = round(auc_baseline, 5),
  auc_GA           = round(resultado_GA_seleccion@fitnessValue, 5),
  ganancia_base_local = round(ganancia_base, 3),
  ganancia_GA_local   = round(ganancia_GA, 3),
  vars_ganadoras   = paste(vars_ganadoras, collapse = "|")
)
fwrite(log_corrida, file = "GA_log_corridas_problema10.txt",
       sep = "\t", append = TRUE)
cat("\nLog de trazabilidad actualizado (GA_log_corridas_problema10.txt), corrida k =", k, "\n")
# --------------------------------------------------------------------




Columnas GA_var finales en el dataset: 13 
    variable
      <char>
 1: GA_var02
 2: GA_var04
 3: GA_var05
 4: GA_var07
 5: GA_var08
 6: GA_var10
 7: GA_var11
 8: GA_var13
 9: GA_var14
10: GA_var15
11: GA_var17
12: GA_var18
13: GA_var20
                                                                                  formula
                                                                                   <char>
 1:                                     div_protegida(cpayroll_trx, Master_mconsumototal)
 2:      (ccallcenter_transacciones + Master_mfinanciacion_limite) * Master_mconsumototal
 3: log_seguro(log_seguro(Master_cconsumos - thomebanking)) * ctarjeta_visa_transacciones
 4:                          raiz_segura(raiz_segura(Master_cconsumos * ctarjeta_master))
 5:           Visa_cconsumos * log_seguro(Master_mpagominimo + mtransferencias_recibidas)
 6:                                                  Visa_mconsumototal + mactivos_margen
 7:             raiz_segura(ctrx_quarter)

##### Fin GA

In [39]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [40]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"                     "clase01"                           
[57] "GA_var02"                           "GA_var04"                          
[59] "GA_var05"                           "GA_var07"                          
[61] "GA_var08"                           "GA_var10"                          
[63] "GA_var11"                           "GA_var13"                          
[65] "GA_var14"                           "GA_var15"                          
[67] "GA_var17"                           "GA_var18"                          
[69] "GA_var20"                           "kmes"                              
[71] "mpayroll_sobre_edad"

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [41]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [42]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [43]:
ncol(dataset)
colnames(dataset)

[1] 343

[1] "numero_de_cliente"                        
  [2] "foto_mes"                                 
  [3] "internet"                                 
  [4] "cliente_edad"                             
  [5] "cliente_antiguedad"                       
  [6] "mrentabilidad"                            
  [7] "mrentabilidad_annual"                     
  [8] "mcomisiones"                              
  [9] "mactivos_margen"                          
 [10] "mpasivos_margen"                          
 [11] "cproductos"                               
 [12] "mcuenta_corriente"                        
 [13] "mcaja_ahorro"                             
 [14] "cdescubierto_preacordado"                 
 [15] "mcuentas_saldo"                           
 [16] "ctarjeta_visa"                            
 [17] "ctarjeta_visa_transacciones"              
 [18] "mtarjeta_visa_consumo"                    
 [19] "ctarjeta_master"                          
 [20] "ctarjeta_master_transacciones"            
 [21] "mtarjeta_master_consumo"                  
 [22] "cprestamos_personales"                    
 [23] "mprestamos_personales"                    
 [24] "cpayroll_trx"                             
 [25] "mpayroll"                                 
 [26] "mttarjeta_visa_debitos_automaticos"       
 [27] "ccomisiones_mantenimiento"                
 [28] "mcomisiones_mantenimiento"                
 [29] "ccomisiones_otras"                        
 [30] "mtransferencias_recibidas"                
 [31] "ccallcenter_transacciones"                
 [32] "thomebanking"                             
 [33] "chomebanking_transacciones"               
 [34] "ctrx_quarter"                             
 [35] "Master_status"                            
 [36] "Master_mfinanciacion_limite"              
 [37] "Master_Fvencimiento"                      
 [38] "Master_msaldototal"                       
 [39] "Master_mlimitecompra"                     
 [40] "Master_fultimo_cierre"                    
 [41] "Master_fechaalta"                         
 [42] "Master_mconsumototal"                     
 [43] "Master_cconsumos"                         
 [44] "Master_mpagominimo"                       
 [45] "Visa_status"                              
 [46] "Visa_mfinanciacion_limite"                
 [47] "Visa_Fvencimiento"                        
 [48] "Visa_msaldototal"                         
 [49] "Visa_mlimitecompra"                       
 [50] "Visa_fultimo_cierre"                      
 [51] "Visa_fechaalta"                           
 [52] "Visa_mconsumototal"                       
 [53] "Visa_cconsumos"                           
 [54] "Visa_mpagominimo"                         
 [55] "clase_ternaria"                           
 [56] "clase01"                                  
 [57] "GA_var02"                                 
 [58] "GA_var04"                                 
 [59] "GA_var05"                                 
 [60] "GA_var07"                                 
 [61] "GA_var08"                                 
 [62] "GA_var10"                                 
 [63] "GA_var11"                                 
 [64] "GA_var13"                                 
 [65] "GA_var14"                                 
 [66] "GA_var15"                                 
 [67] "GA_var17"                                 
 [68] "GA_var18"                                 
 [69] "GA_var20"                                 
 [70] "kmes"                                     
 [71] "mpayroll_sobre_edad"                      
 [72] "internet_lag1"                            
 [73] "cliente_edad_lag1"                        
 [74] "cliente_antiguedad_lag1"                  
 [75] "mrentabilidad_lag1"                       
 [76] "mrentabilidad_annual_lag1"                
 [77] "mcomisiones_lag1"                         
 [78] "mactivos_margen_lag1"                     
 [79] "mpasivos_margen_lag1"                     
 [80] "cproductos_lag1"                          
 [

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [44]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 100% de los CONTINUA

In [45]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [46]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [47]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

In [48]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [49]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 32938

####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [50]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [51]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [52]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 384, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048),
  feature_fraction= c(0.5, 0.8)
)

##### Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 65 minutos
<br> una Analista Jr  debe ser capaz de tolerar estoicamente esta tortura
<br> (y masticar chicle al mismo tiempo)

In [ ]:
# registro a registro calculo la AUC
tb_nueva[,  c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

Mon Sep 07 11:37:40 2026  64, 64, 0.5 niter 12 AUC 0.999998149610837

Mon Sep 07 11:38:09 2026  64, 64, 0.8 niter 226 AUC 0.999997620928219

Mon Sep 07 11:40:01 2026  64, 256, 0.5 niter 507 AUC 0.999998017440182

Mon Sep 07 11:40:36 2026  64, 256, 0.8 niter 295 AUC 0.999998347866819

Mon Sep 07 11:42:00 2026  64, 512, 0.5 niter 309 AUC 0.999998480037473

Mon Sep 07 11:42:48 2026  64, 512, 0.8 niter 409 AUC 0.999998480037473

Mon Sep 07 11:44:38 2026  64, 1024, 0.5 niter 446 AUC 0.999998480037473

Mon Sep 07 11:45:30 2026  64, 1024, 0.8 niter 332 AUC 0.999998678293455

Mon Sep 07 11:47:36 2026  64, 2048, 0.5 niter 512 AUC 0.999998744378782

Mon Sep 07 11:48:21 2026  64, 2048, 0.8 niter 217 AUC 0.999998083525509

Mon Sep 07 11:49:04 2026  128, 64, 0.5 niter 65 AUC 0.999998149610837

Mon Sep 07 11:49:40 2026  128, 64, 0.8 niter 178 AUC 0.999997026160273

Mon Sep 07 11:50:42 2026  128, 256, 0.5 niter 177 AUC 0.999997290501582

Mon Sep 07 11:51:17 2026  128, 256, 0.8 niter 222 AUC 0.9999982

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [ ]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

In [ ]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

In [ ]:
tb_nueva

### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)


dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

##### Final Training Hyperparameters

In [ ]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
final_model <- lgb.train(
  data= dfinal_train,
  param= param_final,
  verbose= -100
)

In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(final_model, "modelo.txt")

In [ ]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(final_model))
archivo_importancia <- "impo.txt"

fwrite( tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [ ]:
# aplico final_model   a dfuture

prediccion <- predict(
  final_model,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

##### Tabla Prediccion

In [ ]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]

# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "utn-2026-virtual-jr"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 100)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  Sys.sleep(30)
  cat(salida, "\n")
}

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")